In [ ]:
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver import Edge

In [ ]:
class MyBrowser(Edge):
    def __init__(self) -> None:

        super().__init__()
        self.maximize_window()
        self.action = ActionChains(self)
        self.wait = WebDriverWait(self, 60)
        self.get("https://www.linkedin.com/search/results/all")

    def xpath_click(self, contains:str, tagname:str="button") -> None:
        self.find_element(By.XPATH, "//{}[text()[contains(.,'{}')]]".format(tagname, contains)).click()
        self.wait.until(EC.presence_of_element_located((By.ID, "search-reusables__filters-bar")))

    def clear_text(self, input) -> None:
        self.action.double_click(input).double_click().send_keys(Keys.DELETE).perform()

    def __del__(self) -> None:
        self.close()
        self.quit()


In [ ]:
browser = MyBrowser()

In [ ]:
search = browser.find_element(By.CLASS_NAME, "search-global-typeahead__input")
browser.clear_text(search)

search.send_keys("talent acquisition" + Keys.ENTER)

[browser.xpath_click(_filter) for _filter in ["People", "2nd", "3rd+"]]

browser.xpath_click("Locations")
location = browser.find_element(By.XPATH, "//input[@placeholder='Add a location']")

for country in ["Estonia"]:
    browser.clear_text(location)
    location.send_keys(country)

    browser.wait.until(EC.element_to_be_clickable((By.XPATH, "//div[@role='listbox']")))
    location.send_keys(Keys.ARROW_DOWN)
    location.send_keys(Keys.ENTER)

browser.xpath_click("Show results", "span")

confirm = EC.presence_of_element_located((By.XPATH, "//div[@role='dialog']"))

for person in browser.find_element(By.XPATH, "//ul[@role='list']").find_elements(By.TAG_NAME, "li"):
    if "Connect" not in person.text: continue

    browser.xpath_click("Connect", "span")
    browser.wait.until(confirm)

    browser.xpath_click("Send without a note", "span")
    browser.wait.until_not(confirm)

In [ ]:
browser.get("https://www.linkedin.com/mynetwork/invitation-manager/sent/")
modal = (By.CLASS_NAME, "artdeco-modal__actionbar")
card = (By.CLASS_NAME, "invitation-card")

for page in range(1, len(browser.find_elements(By.CLASS_NAME, "artdeco-pagination__indicator"))):
    browser.wait.until(EC.presence_of_all_elements_located(card))
    for person in browser.find_elements(*card):
        
        browser.wait.until_not(EC.visibility_of_element_located(modal))
        when = person.find_element(By.CLASS_NAME, "time-badge")

        if any([_ in when.text for _ in ["2 m", "3 m"]]):
            browser.execute_script("arguments[0].click()", person.find_element(By.CLASS_NAME, "artdeco-button__text"))
            browser.wait.until(EC.visibility_of_element_located(modal)).find_element(By.CLASS_NAME, "artdeco-button--primary").click()

    browser.find_element(By.CLASS_NAME, "artdeco-pagination__button--next").click()

In [ ]:
del browser